# 🏆 Final Pipeline with MultiHeirtt Preprocessing

## 📋 Overview

Pipeline này kết hợp:
1. **MultiHeirtt Preprocessing** - Linearize tables thành văn bản tự nhiên
2. **Fine-tuned E5-small Model** - Từ Notebook 5
3. **Semantic Chunking** - Pre-chunked corpus cho các datasets khác
4. **Hybrid Retrieval** - Dense + BM25
5. **Advanced Reranking** - BAAI/bge-reranker-v2-m3

### 🎯 Đặc biệt cho MultiHeirtt:
- **Linearize tables** → Embedding model hiểu được data trong bảng
- **Row-level chunks** → Matching chính xác cho specific lookups
- **Heavy BM25 weight** → Tốt cho numerical matching

---

## 1. Setup & Imports

In [10]:
# Core Libraries
import os
import sys
import json
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from typing import List, Dict
from pathlib import Path

# Embedding & Retrieval
from sentence_transformers import SentenceTransformer
import faiss

# Reranking
from FlagEmbedding import FlagReranker

# BM25
from rank_bm25 import BM25Okapi

# PyTorch
import torch

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')
import logging
logging.disable(logging.CRITICAL)

# Import shared utils
sys.path.insert(0, str(Path('../..') / 'notebook'))
from utils import (
    load_jsonl,
    load_jsonl_data,
    load_prechunked_corpus,
    normalize_scores,
    aggregate_chunk_scores,
    compute_ndcg,
    evaluate_results_df
)

# Check GPU
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    device = 'cuda'
else:
    print("Running on CPU")
    device = 'cpu'

CUDA available: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


In [11]:
# ============================================================================
# CONFIGURATION
# ============================================================================

# Import configuration from config.py
import config

# Set paths
BASE_DIR = Path(config.DATA_DIR)
DATA_DIR = Path(config.DATA_DIR)
MODELS_DIR = Path(config.MODELS_DIR)
OUTPUT_DIR = Path(config.OUTPUT_DIR)
OUTPUT_DIR.mkdir(exist_ok=True)
CHUNKED_CORPUS_DIR = Path(config.CHUNKED_CORPUS_DIR)

# Import main config
CONFIG = config.CONFIG
DATASET_SPECIFIC_CONFIG = config.DATASET_SPECIFIC_CONFIG
QRELS_MAPPING = config.QRELS_MAPPING
DATASETS = config.DATASETS

# Models
EMBEDDING_MODEL = config.EMBEDDING_MODEL
USE_FINETUNED = config.USE_FINETUNED
RERANKER_MODEL = config.RERANKER_MODEL

# E5 prefixes
E5_QUERY_PREFIX = config.E5_QUERY_PREFIX
E5_PASSAGE_PREFIX = config.E5_PASSAGE_PREFIX

# Print configuration summary
config.print_config()


🏆 FINAL PIPELINE WITH MULTIHEIRTT PREPROCESSING

📊 MODELS:
   ✅ Embedding: ..\..\models\e5-small-financerag-finetuned-v3 (FINE-TUNED)
   Reranker: BAAI/bge-reranker-v2-m3

✂️ CHUNKING:
   ✅ Using PRE-CHUNKED corpus (semantic chunking)
   📂 Source: ..\..\data\chunked_corpus
   Aggregation: max

🔍 RETRIEVAL:
   Hybrid: True (alpha=0.6)
   Top-K: retrieval=100, rerank=50, final=10

🎯 DATASET-SPECIFIC OVERRIDES:
   multiheirtt: {'top_k_retrieval': 200, 'top_k_rerank': 80, 'hybrid_alpha': 0.4, 'use_preprocessing': True, 'preprocessing_mode': 'linearized'}
   tatqa: {'top_k_retrieval': 150, 'top_k_rerank': 60, 'hybrid_alpha': 0.5}
   finqa: {'top_k_retrieval': 120, 'hybrid_alpha': 0.55}
   convfinqa: {'top_k_retrieval': 120, 'hybrid_alpha': 0.55}
   financebench: {'hybrid_alpha': 0.7}
   finder: {'hybrid_alpha': 0.65}
   finqabench: {'hybrid_alpha': 0.6}

📊 MULTIHEIRTT PREPROCESSING:
   ✅ Table linearization ENABLED
   Mode: linearized
   - Converts markdown tables to natural language
   - A

## 2. Helper Functions

In [12]:
# ============================================================================
# E5 PREFIX HELPER
# ============================================================================

def add_e5_prefix(text: str, is_query: bool = True) -> str:
    """Add E5 prefix to text"""
    prefix = E5_QUERY_PREFIX if is_query else E5_PASSAGE_PREFIX
    return f"{prefix}{text}"


def hybrid_search_local(query_emb, query_text, faiss_index, bm25, chunk_texts, top_k, alpha=0.6):
    """
    Local hybrid search combining dense + BM25.
    Note: Different from utils.hybrid_search - uses set union for candidates.
    """
    # Dense search
    dense_scores, dense_indices = faiss_index.search(
        query_emb.reshape(1, -1).astype('float32'), 
        min(top_k * 2, faiss_index.ntotal)
    )
    dense_scores = dense_scores[0]
    dense_indices = dense_indices[0]
    
    # BM25 search
    query_tokens = query_text.lower().split()
    bm25_scores = bm25.get_scores(query_tokens)
    bm25_top_indices = np.argsort(bm25_scores)[::-1][:top_k * 2]
    
    # Combine candidates (union of both methods)
    all_indices = set(dense_indices.tolist()) | set(bm25_top_indices.tolist())
    
    # Normalize and combine
    dense_norm = normalize_scores(dense_scores)
    dense_map = {idx: score for idx, score in zip(dense_indices, dense_norm)}
    bm25_max = max(bm25_scores) if len(bm25_scores) > 0 else 1
    bm25_map = {idx: bm25_scores[idx] / bm25_max if bm25_max > 0 else 0 for idx in all_indices}
    
    final_scores = []
    for idx in all_indices:
        d_score = dense_map.get(idx, 0)
        b_score = bm25_map.get(idx, 0)
        final_scores.append((idx, alpha * d_score + (1 - alpha) * b_score))
    
    final_scores.sort(key=lambda x: x[1], reverse=True)
    top_results = final_scores[:top_k]
    
    return [s for _, s in top_results], [i for i, _ in top_results]


print("✅ Helper functions defined")

✅ Helper functions defined


## 3. Load Models

In [13]:
print("Loading models...")

# 1. Embedding model
print(f"\n1. Loading embedding model: {CONFIG['embedding_model']}")
if CONFIG['use_finetuned']:
    print("   🎯 Using FINE-TUNED model!")
else:
    print("   ⚠️ Using base model")

embed_model = SentenceTransformer(CONFIG['embedding_model'], device=device)
print("   ✅ Done")

# 2. Reranker
print(f"\n2. Loading reranker: {CONFIG['reranker_model']}")
reranker = FlagReranker(CONFIG['reranker_model'], use_fp16=(device=='cuda'))
print("   ✅ Done")

print("\n✅ All models loaded!")

Loading models...

1. Loading embedding model: ..\..\models\e5-small-financerag-finetuned-v3
   🎯 Using FINE-TUNED model!
   ✅ Done

2. Loading reranker: BAAI/bge-reranker-v2-m3
   ✅ Done

✅ All models loaded!


## 4. Main Pipeline Function

In [14]:
def process_dataset(dataset_name: str, config: Dict, dataset_overrides: Dict = None):
    """
    Process dataset with full pipeline:
    1. Load data (preprocessed for MultiHeirtt, pre-chunked for others)
    2. Encode with E5 model
    3. Hybrid search (Dense + BM25)
    4. Rerank with BGE
    5. Evaluate against qrels
    """
    print(f"\n{'='*70}")
    print(f"📊 Processing: {dataset_name.upper()}")
    print(f"{'='*70}")
    
    # Apply overrides
    effective_config = config.copy()
    if dataset_overrides and dataset_name in dataset_overrides:
        overrides = dataset_overrides[dataset_name]
        effective_config.update(overrides)
        print(f"🔧 Overrides: {overrides}")
    
    # Load data
    corpus_df, queries_df, qrels_df = load_jsonl_data(dataset_name, effective_config['data_dir'])
    
    # Prepare chunks
    all_chunks = []
    chunk_to_doc = {}
    
    # MultiHeirtt: Load preprocessed corpus
    if dataset_name == 'multiheirtt' and effective_config.get('use_preprocessing', False):
        preprocessing_mode = effective_config.get('preprocessing_mode', 'linearized')
        preprocessed_file = OUTPUT_DIR / f'multiheirtt_corpus_{preprocessing_mode}.jsonl'
        
        print(f"\n📂 Loading PREPROCESSED MultiHeirtt corpus...")
        print(f"   File: {preprocessed_file}")
        
        if not preprocessed_file.exists():
            raise FileNotFoundError(f"Preprocessed file not found: {preprocessed_file}")
        
        all_chunks = load_jsonl(preprocessed_file)
        for chunk in all_chunks:
            chunk_to_doc[chunk['_id']] = chunk.get('original_id', chunk['_id'])
        
        print(f"   ✅ Loaded {len(all_chunks)} chunks")
    
    # Other datasets: use pre-chunked
    elif effective_config.get('use_prechunked', False):
        print(f"\n📂 Loading pre-chunked corpus...")
        result = load_prechunked_corpus(dataset_name, effective_config['chunked_corpus_dir'])
        if result and result[0]:
            all_chunks = result[0]
            for c in all_chunks:
                chunk_id = c.get('_id', c.get('chunk_id', ''))
                chunk_to_doc[chunk_id] = c.get('original_id', c.get('doc_id', chunk_id))
    
    # Fallback: use original
    if not all_chunks:
        print(f"\n⚠️ Using original corpus...")
        for _, row in corpus_df.iterrows():
            doc_id = str(row['_id'])
            text = f"{row.get('title', '')} {row.get('text', '')}".strip()
            all_chunks.append({'_id': doc_id, 'text': text, 'original_id': doc_id})
            chunk_to_doc[doc_id] = doc_id
    
    # Prepare texts
    chunk_texts_raw = [c.get('text', '')[:1024] for c in all_chunks]
    chunk_texts = [add_e5_prefix(t, is_query=False) for t in chunk_texts_raw]
    chunk_ids = [c.get('_id', '') for c in all_chunks]
    
    # Encode chunks
    print(f"\n🔢 Encoding {len(chunk_texts)} chunks...")
    chunk_embeddings = embed_model.encode(
        chunk_texts,
        batch_size=effective_config['embed_batch_size'],
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    
    # Build FAISS index
    print(f"\n🔍 Building FAISS index...")
    index = faiss.IndexFlatIP(chunk_embeddings.shape[1])
    index.add(chunk_embeddings.astype('float32'))
    print(f"   ✅ {index.ntotal} vectors")
    
    # Build BM25
    bm25 = None
    if effective_config['use_hybrid']:
        print(f"\n🔤 Building BM25 index...")
        tokenized = [t.lower().split() for t in chunk_texts_raw]
        bm25 = BM25Okapi(tokenized)
        print(f"   ⚖️ Alpha: {effective_config['hybrid_alpha']}")
    
    del chunk_embeddings
    if device == 'cuda':
        torch.cuda.empty_cache()
    
    # Encode queries
    print(f"\n🎯 Processing {len(queries_df)} queries...")
    query_texts_raw = [str(r.get('text', '')) for _, r in queries_df.iterrows()]
    query_texts = [add_e5_prefix(t, is_query=True) for t in query_texts_raw]
    query_ids = queries_df['_id'].tolist()
    
    query_embeddings = embed_model.encode(
        query_texts,
        batch_size=effective_config['embed_batch_size'],
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    
    # Retrieve & Rerank
    print(f"\n🔎 Retrieving & Reranking...")
    results = []
    
    for i, query_id in enumerate(tqdm(query_ids, desc="Retrieve+Rerank")):
        query_emb = query_embeddings[i]
        query_text = query_texts_raw[i]
        
        # Hybrid search
        if effective_config['use_hybrid'] and bm25:
            scores, chunk_indices = hybrid_search_local(
                query_emb, query_text, index, bm25, chunk_texts_raw,
                effective_config['top_k_retrieval'], effective_config['hybrid_alpha']
            )
        else:
            scores, chunk_indices = index.search(
                query_emb.reshape(1, -1).astype('float32'),
                effective_config['top_k_retrieval']
            )
            scores, chunk_indices = scores[0].tolist(), chunk_indices[0].tolist()
        
        # Aggregate chunks to docs
        doc_scores = {}
        for idx, score in zip(chunk_indices, scores):
            if idx < 0 or idx >= len(chunk_ids):
                continue
            doc_id = chunk_to_doc.get(chunk_ids[idx], chunk_ids[idx])
            if doc_id not in doc_scores:
                doc_scores[doc_id] = []
            doc_scores[doc_id].append(float(score))
        
        doc_agg = aggregate_chunk_scores(doc_scores, effective_config['chunk_aggregation'])
        sorted_docs = sorted(doc_agg.items(), key=lambda x: x[1], reverse=True)[:effective_config['top_k_rerank']]
        
        # Rerank
        candidate_ids = [d[0] for d in sorted_docs]
        candidate_texts = []
        for doc_id in candidate_ids:
            doc_row = corpus_df[corpus_df['_id'] == doc_id]
            text = str(doc_row['text'].values[0])[:2048] if len(doc_row) > 0 else ""
            candidate_texts.append(text)
        
        pairs = [[query_text, t] for t in candidate_texts]
        rerank_scores = reranker.compute_score(pairs)
        if not isinstance(rerank_scores, list):
            rerank_scores = [rerank_scores]
        
        scored = sorted(zip(candidate_ids, rerank_scores), key=lambda x: x[1], reverse=True)
        
        for doc_id, score in scored[:effective_config['top_k_final']]:
            results.append({'query_id': query_id, 'corpus_id': doc_id, 'score': float(score)})
    
    results_df = pd.DataFrame(results)
    print(f"   ✅ Generated {len(results_df)} results")
    
    # Evaluate
    eval_metrics = {}
    if effective_config['eval_on_qrels'] and qrels_df is not None:
        print(f"\n📊 Evaluating...")
        eval_metrics = evaluate_results_df(results_df, qrels_df)
        print(f"   NDCG@10: {eval_metrics['NDCG@10']:.4f}")
    
    # Cleanup
    del query_embeddings, index
    if device == 'cuda':
        torch.cuda.empty_cache()
    
    return results_df, eval_metrics


print("✅ Pipeline function defined")

✅ Pipeline function defined


## 5. Run Complete Pipeline

In [15]:
all_results = []
all_eval = {}
failed = []

print("\n" + "="*70)
print("🚀 STARTING FINAL PIPELINE WITH MULTIHEIRTT PREPROCESSING")
print("="*70)

print("\n📋 Configuration:")
print(f"   Model: {CONFIG['embedding_model']}")
print(f"   Fine-tuned: {CONFIG['use_finetuned']}")
print(f"   Hybrid: {CONFIG['use_hybrid']}")

print("\n🔧 Dataset-specific settings:")
for ds in CONFIG['datasets']:
    if ds in DATASET_SPECIFIC_CONFIG:
        print(f"   {ds}: {DATASET_SPECIFIC_CONFIG[ds]}")

for dataset in CONFIG['datasets']:
    try:
        df_res, metrics = process_dataset(
            dataset,
            CONFIG,
            DATASET_SPECIFIC_CONFIG
        )
        all_results.append(df_res)
        if metrics:
            all_eval[dataset] = metrics
    except Exception as e:
        print(f"\n❌ Error processing {dataset}: {e}")
        import traceback
        traceback.print_exc()
        failed.append(dataset)

print(f"\n{'='*70}")
print(f"✅ Pipeline completed: {len(all_results)}/{len(CONFIG['datasets'])} datasets")
if failed:
    print(f"❌ Failed: {failed}")
print(f"{'='*70}")


🚀 STARTING FINAL PIPELINE WITH MULTIHEIRTT PREPROCESSING

📋 Configuration:
   Model: ..\..\models\e5-small-financerag-finetuned-v3
   Fine-tuned: True
   Hybrid: True

🔧 Dataset-specific settings:
   convfinqa: {'top_k_retrieval': 120, 'hybrid_alpha': 0.55}
   financebench: {'hybrid_alpha': 0.7}
   finder: {'hybrid_alpha': 0.65}
   finqa: {'top_k_retrieval': 120, 'hybrid_alpha': 0.55}
   finqabench: {'hybrid_alpha': 0.6}
   multiheirtt: {'top_k_retrieval': 200, 'top_k_rerank': 80, 'hybrid_alpha': 0.4, 'use_preprocessing': True, 'preprocessing_mode': 'linearized'}
   tatqa: {'top_k_retrieval': 150, 'top_k_rerank': 60, 'hybrid_alpha': 0.5}

📊 Processing: CONVFINQA
🔧 Overrides: {'top_k_retrieval': 120, 'hybrid_alpha': 0.55}
  Loaded 2066 docs, 421 queries

📂 Loading pre-chunked corpus...
  ✅ Loaded 38909 pre-chunked chunks
  📋 Method used: unknown

🔢 Encoding 38909 chunks...


Batches:   0%|          | 0/1216 [00:00<?, ?it/s]


🔍 Building FAISS index...
   ✅ 38909 vectors

🔤 Building BM25 index...
   ⚖️ Alpha: 0.55

🎯 Processing 421 queries...


Batches:   0%|          | 0/14 [00:00<?, ?it/s]


🔎 Retrieving & Reranking...


...


🔍 Building FAISS index...
   ✅ 21196 vectors

🔤 Building BM25 index...
   ⚖️ Alpha: 0.5

🎯 Processing 1663 queries...


Batches:   0%|          | 0/52 [00:00<?, ?it/s]


🔎 Retrieving & Reranking...


Retrieve+Rerank:   0%|          | 0/1663 [00:00<?, ?it/s]

   ✅ Generated 16630 results

📊 Evaluating...
   NDCG@10: 0.5044

✅ Pipeline completed: 7/7 datasets


## 6. Evaluation Summary

In [16]:
if all_eval:
    print("\n" + "="*70)
    print("📊 EVALUATION SUMMARY (NDCG@10)")
    print("="*70)
    
    total_ndcg = 0
    total_queries = 0
    
    for ds, m in sorted(all_eval.items()):
        marker = "🔧" if ds == 'multiheirtt' else "  "
        print(f"\n{marker} {ds.upper():15s}: {m['NDCG@10']:.4f} ({m['num_queries']} queries)")
        total_ndcg += m['NDCG@10'] * m['num_qrels']
        total_queries += m['num_qrels']
    
    if total_queries > 0:
        avg_ndcg = total_ndcg / total_queries
        print(f"\n{'='*70}")
        print(f"📈 WEIGHTED AVERAGE NDCG@10: {avg_ndcg:.4f}")
        print(f"{'='*70}")
else:
    print("\n⚠️ No evaluation metrics available")


📊 EVALUATION SUMMARY (NDCG@10)

   CONVFINQA      : 0.4971 (421 queries)

   FINANCEBENCH   : 0.7369 (150 queries)

   FINDER         : 0.3412 (216 queries)

   FINQA          : 0.4427 (1147 queries)

   FINQABENCH     : 0.8773 (100 queries)

🔧 MULTIHEIRTT    : 0.1556 (974 queries)

   TATQA          : 0.5044 (1663 queries)

📈 WEIGHTED AVERAGE NDCG@10: 0.4238


## 7. Generate Submission

In [17]:
if all_results:
    final_df = pd.concat(all_results, ignore_index=True)
    submission_df = final_df[['query_id', 'corpus_id']]
    
    # Save
    submission_df.to_csv(CONFIG['output_file'], index=False)
    
    print(f"\n✅ Submission saved: {CONFIG['output_file']}")
    print(f"   Total entries: {len(submission_df):,}")
    print(f"   Unique queries: {submission_df['query_id'].nunique():,}")
    
    print(f"\n📋 Sample results:")
    print(submission_df.head(15))
    
    # Validation
    counts = submission_df.groupby('query_id').size()
    print(f"\n🔍 Validation:")
    print(f"   Results per query: {dict(counts.value_counts().sort_index())}")
    
    if (counts == 10).all():
        print(f"   ✅ All queries have exactly 10 results")
    else:
        print(f"   ⚠️ Some queries don't have 10 results")
else:
    print("\n❌ No results to save")


✅ Submission saved: output\submission_with_multiheirtt_preprocessing.csv
   Total entries: 46,710
   Unique queries: 4,671

📋 Sample results:
     query_id  corpus_id
0   qd4982518  dd4c4f7aa
1   qd4982518  dd4b88920
2   qd4982518  dd4bb016e
3   qd4982518  dd4b9f7f6
4   qd4982518  dd4b87d18
5   qd4982518  dd4bec0ec
6   qd4982518  dd4be45d6
7   qd4982518  dd4bd3790
8   qd4982518  dd4b89cbc
9   qd4982518  dd4972b86
10  qd49795a8  dd4befb5c
11  qd49795a8  dd4c05bc8
12  qd49795a8  dd4bd7b9c
13  qd49795a8  dd4979602
14  qd49795a8  dd4b9e8ce

🔍 Validation:
   Results per query: {10: 4671}
   ✅ All queries have exactly 10 results


## 8. Final Summary

In [18]:
print("\n" + "="*70)
print("🏆 FINAL PIPELINE COMPLETED!")
print("="*70)

print("\n✅ Key Features:")
print("   1. 🧠 Fine-tuned E5-small model")
print("   2. ✨ Semantic chunking (optimal per dataset)")
print("   3. 📊 MultiHeirtt TABLE PREPROCESSING")
print("      → Linearized tables from preprocess_corpus.ipynb")
print("      → +9.3% improvement over original")
print("   4. 🔍 Hybrid search (Dense + BM25)")
print("   5. 🎯 BGE-reranker-v2-m3")
print("   6. 📝 E5 prefixes enabled")

print("\n🎯 Dataset optimizations:")
print("   - MultiHeirtt: Preprocessed corpus + top_k=200 + 60% BM25")
print("   - TATQA: top_k=150, balanced hybrid")
print("   - FinQA/ConvFinQA: top_k=120, slight BM25 boost")

if all_eval:
    total_ndcg = sum(m['NDCG@10'] * m['num_qrels'] for m in all_eval.values())
    total_queries = sum(m['num_qrels'] for m in all_eval.values())
    if total_queries > 0:
        avg = total_ndcg / total_queries
        print(f"\n📊 Final NDCG@10: {avg:.4f}")
        if 'multiheirtt' in all_eval:
            print(f"   MultiHeirtt NDCG@10: {all_eval['multiheirtt']['NDCG@10']:.4f}")

print(f"\n💾 Output: {CONFIG['output_file']}")
print(f"\n🚀 Ready for submission!")
print("="*70)


🏆 FINAL PIPELINE COMPLETED!

✅ Key Features:
   1. 🧠 Fine-tuned E5-small model
   2. ✨ Semantic chunking (optimal per dataset)
   3. 📊 MultiHeirtt TABLE PREPROCESSING
      → Linearized tables from preprocess_corpus.ipynb
      → +9.3% improvement over original
   4. 🔍 Hybrid search (Dense + BM25)
   5. 🎯 BGE-reranker-v2-m3
   6. 📝 E5 prefixes enabled

🎯 Dataset optimizations:
   - MultiHeirtt: Preprocessed corpus + top_k=200 + 60% BM25
   - TATQA: top_k=150, balanced hybrid
   - FinQA/ConvFinQA: top_k=120, slight BM25 boost

📊 Final NDCG@10: 0.4238
   MultiHeirtt NDCG@10: 0.1556

💾 Output: output\submission_with_multiheirtt_preprocessing.csv

🚀 Ready for submission!
